# Aggregate Results from `ham_parse_bc.py`

In this notebook, cellBC[bc][umi] = count python dictionary outputs of `ham_parse_bc.py` are going to be processed to cellBC x bc `csv` files. In particular, I will walk you through

1. Merge dictionary outputs of same samples sequenced on different lanes.

2. Filter out data derived from sequencing error.

3. Correct raw cellBC using the cellranger bam information.

4. Convert the result into a cellBC x bc dataframe ready to use for the next steps.

In [1]:
import pandas as pd
import numpy as np
import pickle
from collections import defaultdict
import os
import csv
import seaborn as sns
import matplotlib.pyplot as plt

## 1. Merge Dictionaries

In [371]:
# Example files to use. These should be the output of `ham_parse_bc.py`

filename1 = 'ham_JCF-CBC-1_S22_L007'
filename2 = 'ham_JCF-CBC-1_S22_L008'

In [86]:
# the saved dictionaries should take the structure of cellBC[bc][umi] = count

def merge_dicts(dict1, dict2):
    merged = defaultdict(lambda: defaultdict(dict))  # Nested defaultdict to handle missing keys

    for cellBC, bc_dict in dict1.items():
        for bc, umi_dict in bc_dict.items():
            for umi, count in umi_dict.items():
                merged[cellBC][bc][umi] = count

    for cellBC, bc_dict in dict2.items():
        for bc, umi_dict in bc_dict.items():
            for umi, count in umi_dict.items():
                if umi in merged[cellBC][bc]:  # if this same umi(molecule) is already captured in one reading split, then merge that into the count (this umi is not a new unique one)
                    merged[cellBC][bc][umi] += count  # Summing counts if umi exists
                else:
                    merged[cellBC][bc][umi] = count  

    return dict(merged)  

In [372]:
# Load pickled dictionaries
with open('../6-grammar_final/raw_dict/' + filename1 + ".pkl", "rb") as f:
    dict1 = pickle.load(f)

with open('../6-grammar_final/raw_dict/' + filename2 + ".pkl", "rb") as f:
    dict2 = pickle.load(f)

# Merge them
merged_dict = merge_dicts(dict1, dict2)


In [14]:
# convert the dictionary to human readable csv file, still keep the umi and count

def convert_to_matrix(cellBC_dict, filename, result_path='../6-grammar_final/'):
    output_file = os.path.join(result_path, filename)
    with open(output_file, 'w') as f:
        writer = csv.writer(f)
        writer.writerow(['cellBC', 'bc', 'umi', 'count'])

        for cellBC, bc_dict in cellBC_dict.items():
            for BC, umi_dict in bc_dict.items():
                for umi, count in umi_dict.items():
                    writer.writerow([cellBC, BC, umi, count])

In [373]:
convert_to_matrix(merged_dict, 'JCF-CBC-1.csv')

## 2. Filter out reads with low counts

In [374]:
df = pd.read_csv('../6-grammar_final/JCF-CBC-1.csv')

In [ ]:
cutoff = 40 # 40 for cbc, 30 for ebc

In [377]:
df = df[df['count'] > cutoff].reset_index(drop = True)

## 3. Correct for the cellBC

Now use the `.bam` output of `cellranger count` to correct some cell barcode sequences that arise from sequencing error. The logic is like this: when running the cellranger count, the 10x algorithm automatically correct cell barcodes based on their cellBC whitelist and the sequencing quality. And that information is kept in a `.bam` file with original barcode -> whitelist barcode. 

Use the `bam.csv` output from running `bam_to_csv.sh`.

In [380]:
count_pwd = '../6-grammar_final/bams/'
bc = pd.read_csv(count_pwd + 'JCF-CBC-1_bam.csv')

In [382]:
# remove any raw bc that can be mapped to multiple true cellBC
bc = bc[~bc.iloc[:, 0].duplicated(keep=False)]

In [383]:
bc = bc.rename(columns={'0': 'seq_cellBC', '1': 'map_cellBC'})

In [385]:
# Inner merge the output of bc parsing and the cellBC correction map
# Drop any cellBC that only appears in one of the dataframe. 
df = df.merge(bc, left_on = 'cellBC', right_on = 'seq_cellBC', how = 'inner')

In [387]:
# convert cellBC to mapBC (that will be the same as found in the adata object)
df['cellBC'] = df['map_cellBC'].str.split('-').str[0]

In [388]:
df = df.drop(columns=['seq_cellBC', 'map_cellBC'])

In [390]:
# now, some cellBC will be corrected, thus there will be duplicated [cellBC, bc] combination
# aggregate on that
df = df.groupby(['cellBC', 'bc'], as_index=False)[['umi']].nunique()

## 4. Convert to pivot table

In [392]:
# now, to better be able to fit into the following pipeline, and to make it more readable
# convert to a pivot matrix
df = df.pivot(index = 'cellBC', columns = 'bc', values = 'umi').fillna(0)

In [51]:
# if not all BC are found, add a column and fill it with 0
lookup = pd.read_csv('../6-grammar_final/refs/supp9_enhancer_ebc_cbc_lookup.csv')

In [394]:
# depends on whether you are processing cbc or ebc, change the column name to look at
column_name = 'cbc' # 'cbc' or 'ebc'

In [395]:
required_columns = lookup[column_name].tolist()
missing_columns = set(required_columns) - set(df.columns)

# Add missing columns with all entries set to 0
for col in missing_columns:
    df[col] = 0

In [396]:
# check for number of missing columns (how many ordered barcodes we are unable to find)
# there should only be one or two
missing_columns

set()

## END and SAVE

In [3]:
df

,cellBC,AAAAACGTGAACGGGCGTTAGGCCG,AAAAGAGTCAACGCACGTTACTAAC,AAACAGCACGGCGTGAGCCGGAGTA,AAACCAGTCACAGGCGCGCACCCTA,AAAGCGGGGCGAACAGCACAGTCCG,AAAGCTTTGATCAAGCACCTAGGCG,AAATAACAACCAGCAGCGCAACGTA,AAATCTAACCACCTACCCCTCGTGC,AACACGGACGGTCTTGATGGCCGAA,...,GTGAGGGGCTCTCTGGCCGAGTATA,GTGGCCTCGATGAGACCCACCGTTA,GTGGCTAAACACGAGTCGGCCCGAC,GTGGGTAAGTTCACGACCCTGAGAA,GTTAAGGACGTGATGCCAGTCTCCG,GTTACTGGCATGCTTAGGGTCCTCA,GTTGAATAAGATCTTCGGTGGCCAA,GTTGAGAGGCTTGGTGACTGATTAC,GTTGCGCGATGGACGAATGAAGAGA,GTTGGCCTCGAGCAAGGCAAGGTTA
0,AAACCAAAGAATCGTT,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,AAACCAAAGATTGACT,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,AAACCAAAGCAACCAC,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,AAACCAAAGCATGACC,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,AAACCAAAGCCAACTA,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
98866,TGTGTTGAGTCAATCG,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
98867,TGTGTTGAGTCGACGT,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
98868,TGTGTTGAGTGATGGG,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
98869,TGTGTTGAGTTACCGA,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [398]:
df.to_csv('../6-grammar_final/JCF-CBC-1.csv')